IMPORTANT! Before beginning any lab assignment, be sure to **make your own copy** of the notebook and name it "lastname - Lab 11" or something similar.


# Lab 11: Reinforcement Learning with Gymnasium and Stable Baselines 3

In lecture we studied the core concepts of reinforcement learning: agents, environments, states, actions, rewards, policies, and the algorithms (Q-learning, DQN, PPO) that an agent uses to learn through trial and error.

In this lab you will move from theory to practice by working directly with the two most important RL Python libraries:

- **Gymnasium** provides a standardized API for interacting with RL environments - both built-in classics and custom ones you write yourself.
- **Stable Baselines 3 (SB3)** provides production-quality implementations of deep RL algorithms (DQN, PPO, and others) under a consistent, scikit-learn-like API.

The lab is organized into three activities:

- **Part A**: Build a custom Gymnasium environment - a reimplementation of the 7x7 fire-grid world from lecture.
- **Part B**: Apply DQN to CartPole and see how the **exploration fraction** hyperparameter shapes what the agent learns.
- **Part C**: Apply PPO to CartPole and see how the **discount factor** $\gamma$ hyperparameter shapes what the agent learns.


### Installation of Gymnasium and Stable Baselines 3

An ancient version of Gymnasium is pre-installed in Colab, but it is too old to be compatible with Stable Baselines 3, so the cell below installs both. It also installs:

- `swig` and `gymnasium[box2d]` - needed for physics-based environments like LunarLander, and
- `ffmpeg` - needed to export agent videos (used at the end of Part C).


In [ ]:
# Install required packages for reinforcement learning (uninstalling the
# out of date OpenAI Gymnasium and replacing it with the Farama version)
!pip uninstall gym -y
!pip install --upgrade gymnasium
!pip install gymnasium[box2d]  
!pip install stable-baselines3
!apt-get install -y -q ffmpeg  # For video recording and export

Let's test if the installs worked by importing Gymnnasium and Stable Baselines 3. If you don't get any errors, you're good to go!


In [ ]:
import gymnasium as gym
import stable_baselines3 as sb3
print(f'gymnasium         {gym.__version__}')
print(f'stable-baselines3 {sb3.__version__}')
print('All packages ready.')

Now let's load all the libraries we'll need for the lab.


In [ ]:
# Standard imports used throughout the lab
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from pathlib import Path

SEED = 42
np.random.seed(SEED)
print('Imports complete.')

### The Gymnasium API

Every Gymnasium environment - whether a built-in classic or a custom one you write - exposes the same interface. This standardization is what makes it easy to swap environments and plug any SB3 algorithm into any Gymnasium environment without changing your training code.

| Call                                                          | What it does                                             |
| ------------------------------------------------------------- | -------------------------------------------------------- |
| `env = gym.make('Name')`                                      | Create an environment instance                           |
| `obs, info = env.reset()`                                     | Reset to the initial state; return the first observation |
| `obs, reward, terminated, truncated, info = env.step(action)` | Take one action                                          |
| `action = env.action_space.sample()`                          | Sample a uniformly random valid action                   |
| `env.close()`                                                 | Release resources                                        |

Two flags signal episode end: `terminated` means the agent reached a natural terminal state (e.g., the goal or a crash), while `truncated` means the episode hit a step-count time limit. In either case you should call `env.reset()` before starting the next episode.

The cell below demonstrates this interface on the built-in `CartPole-v1` environment using a random policy. This environment is a classic control problem where the agent must balance a pole on a cart by moving left or right. The episode ends when the pole falls too far (|θ| > 12 degrees) or after 500 steps. The reward is +1 for every step the pole remains balanced, so the maximum episode reward is 500.


In [ ]:
# Explore CartPole-v1 using the Gymnasium API
env = gym.make('CartPole-v1')

# Describe the environment and its reward structure
print('CartPole-v1 environment')
print(f'  Observation space: {env.observation_space}')
# Translate the observation space bounds into human-readable parameter 
# names and ranges
print(' which corresponds to:')
param_names = ['cart position (x)', 'cart velocity (v)', 
               'pole angle (θ)', 'pole angular velocity (ω)']
for i, param in enumerate(param_names):
    low = env.observation_space.low[i]
    high = env.observation_space.high[i]
    print(f'    {param}: [{low:.2f}, {high:.2f}]')
print(f'  Action space:      {env.action_space}  (0=push left, 1=push right)')
print(f'  Reward structure:  +1 every timestep the pole remains upright')
print(f'  Max return per episode: 500')
print()

# Run one full episode with a random policy
# The `reset()` method returns the initial observation and an info dictionary. 
obs, info = env.reset(seed=SEED)
# Print the initial observation (rounded for readability)
print(f'Initial observation (x, v, θ, ω): {np.round(obs, 3)}')
# and the info dictionary (if you want to see it).
print(f'Info dictionary: {info}')
print()

# Now let's run one full episode using a random policy.
total_reward = 0
step = 0
done = False
while not done:
    # Choose a random action from the action space
    action = env.action_space.sample()
    # Take the action and observe the results
    obs, reward, terminated, truncated, info = env.step(action)
    # Accumulate the reward
    total_reward += reward
    # Check if the episode has ended (either terminated or truncated)
    done = terminated or truncated
    # Count the step and print details for the first few steps
    step += 1
    # Print details for first 5 steps, then summarize the rest
    if step <= 5:                                          
        this_action = 'push left' if action == 0 else 'push right'
        print(f'  Step {step}: action={action} ({this_action}), '
              f'reward={reward:.0f}, done={done}')

if step > 5:
    print(f'  ... ({step - 5} more steps) ...')
print(f'\nEpisode finished in {step} steps,  total reward = {total_reward:.0f}')
env.close()

**Question 1**: A random agent on CartPole typically survives only 10-25 steps before the pole falls. What does this tell you about the baseline a learning algorithm must beat, and why do random actions fail so badly at this specific task?


```
ANSWER HERE
```


## Part A: Building a Custom Gymnasium Environment

### Background: The Five Requirements

You can apply any SB3 algorithm to your own problem by subclassing `gym.Env`. Your class only needs to implement five things:

1. `__init__()` - define `self.action_space`, `self.observation_space`, and any fixed parameters.
2. `reset()` - return the agent to the start state; return `(observation, info_dict)`.
3. `step(action)` - advance by one timestep; return `(obs, reward, terminated, truncated, info_dict)`.
4. `render()` - optional visualization (can return `None` or an image array).
5. `close()` - optional resource cleanup.

The two most important attributes set in `__init__` are:

- `self.action_space` - what actions the agent can take. `spaces.Discrete(n)` gives $n$ integers $\{0, 1, \ldots, n-1\}$.
- `self.observation_space` - what the agent sees. `spaces.Discrete(n)` for integer states; `spaces.Box(low, high, shape)` for continuous vectors.


### The 7x7 Fire Grid World

![Figure 11.4: The 7x7 fire grid world.](https://raw.githubusercontent.com/JuanCab/csis446_lab11/refs/heads/main/images/fmlpda_figure_11_4.png)

_Figure 11.4 from Kelleher, MacNamee, and D'Arcy (2020)._

Recall the fire grid world from lecture. We will wrap the fire grid world from lecture as a proper Gymnasium environment. Remember that for the fire grid world:

- We used a 7x7 grid; the agent starts at position (row 0, column 2), marked **S**.
- The goal is at (row 6, column 4), marked **G**. Reaching it gives reward $+50$ and ends the episode.
- Six fire cells at rows 2-4, columns 2 and 4, marked **F**. Entering one gives reward $-20$ but the episode continues.
- Every other step gives reward $-1$ (encouraging the agent to find a short path).
- Four actions: up (0), down (1), left (2), right (3). Moving into a wall leaves the agent in place.
- State encoding is state $= \text{row} \times 7 + \text{col}$, giving 49 integer states.


### Step A.1: The Environment Class - Worked Example

Read through the class below carefully. Each method is annotated to explain its role in the Gymnasium contract. There are some questions after this to test your understanding of how the environment works.


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

# Create a custom environment by subclassing gym.Env and implementing
# the required methods.
class FireGridWorldEnv(gym.Env):
    """
    7x7 deterministic grid world with fire penalties and a goal state.

    Reimplements the textbook fire grid world as a Gymnasium-compliant
    custom environment.
    """

    # metadata tells Gymnasium which render modes we support
    # (we only implement a simple text-based 'ansi' mode here)
    # Some environments support multiple render modes (e.g.,
    # 'human' for a GUI, 'rgb_array' for pixel arrays, etc.)
    metadata = {'render_modes': ['ansi']}

    # -------------------------------------------------------------- #
    # METHOD 1 - __init__                                            #
    # Define spaces and store any fixed environment parameters.      #
    # Called once when you create the environment.                   #
    # -------------------------------------------------------------- #
    def __init__(self, grid_size=(7, 7), start=(0, 2), goal=(6, 4),
                 fire_cells=[(2,2),(3,2),(4,2),(2,4),(3,4),(4,4)],
                 max_steps=300,
                 fire_penalty=-20.0, step_penalty=-1.0, goal_reward=50.0):
        super().__init__()
        self.grid_size    = grid_size
        self.start        = start
        self.goal         = goal
        self.fire_cells   = set(fire_cells or [])
        self.max_steps    = max_steps
        # reward for plain (non-fire, non-goal) move
        self.step_penalty = step_penalty
        self.fire_penalty = fire_penalty
        self.goal_reward  = goal_reward

        # Action space: 4 discrete actions  (up=0, down=1, left=2, right=3)
        self.action_space = spaces.Discrete(4)

        # Observation space: one integer per grid cell (0 to 48 for a 7x7 grid)
        self.observation_space = spaces.Discrete(grid_size[0] * grid_size[1])

        # Internal mutable state
        self._pos   = start
        self._steps = 0

    # Helper: convert (row, col) tuple to a single integer observation
    def _to_obs(self, pos):
        return pos[0] * self.grid_size[1] + pos[1]

    # -------------------------------------------------------------- #
    # METHOD 2 - reset                                               #
    # Called at the beginning of every episode.                      #
    # Must return (observation, info_dict).                          #
    # -------------------------------------------------------------- #
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)    # handles seeding of self.np_random
        self._pos   = self.start
        self._steps = 0
        return self._to_obs(self._pos), {}

    # -------------------------------------------------------------- #
    # METHOD 3 - step                                                #
    # Execute one action and return the transition result.           #
    # Must return (obs, reward, terminated, truncated, info_dict).   #
    # -------------------------------------------------------------- #
    def step(self, action):
        r, c = self._pos
        rows, cols = self.grid_size

        # Apply movement; clip to grid boundaries (agent cannot walk off the edge)
        if   action == 0:  r = max(r - 1, 0)          # up
        elif action == 1:  r = min(r + 1, rows - 1)   # down
        elif action == 2:  c = max(c - 1, 0)          # left
        elif action == 3:  c = min(c + 1, cols - 1)   # right

        self._pos    = (r, c)
        self._steps += 1

        # Assign reward and check for natural episode termination
        if self._pos == self.goal:
            reward     = self.goal_reward
            terminated = True    # episode ends naturally: goal reached
        elif self._pos in self.fire_cells:
            reward     = self.fire_penalty
            terminated = False   # fire is a penalty but NOT episode end
        else:
            reward     = self.step_penalty
            terminated = False

        # Truncate when the step budget is exhausted
        truncated = (self._steps >= self.max_steps)

        return self._to_obs(self._pos), reward, terminated, truncated, {}

    # --------------------------------------------------------------- #
    # METHOD 4 - render                                               #
    # Optional human-readable visualization of the current state.     #
    # --------------------------------------------------------------- #
    def render(self):
        rows, cols = self.grid_size
        lines = []
        for r in range(rows):
            row = ''
            for c in range(cols):
                if   (r, c) == self._pos:        row += ' A '  # agent
                elif (r, c) == self.goal:        row += ' G '  # goal
                elif (r, c) in self.fire_cells:  row += ' F '  # fire
                elif (r, c) == self.start:       row += ' S '  # start
                else:                            row += ' . '  # empty
            lines.append(row)
        return '\n'.join(lines)

    # --------------------------------------------------------------- #
    # METHOD 5 - close                                                #
    # Called when the environment is no longer needed.                #
    # --------------------------------------------------------------- #
    def close(self):
        pass   # nothing to clean up for this simple environment


# --- Shared constants used throughout the lab ---
FIRE_CELLS  = [(2,2),(3,2),(4,2),(2,4),(3,4),(4,4)]
GRID_SIZE   = (7, 7)
START_STATE = (0, 2)
GOAL_STATE  = (6, 4)
FIRE_PENALTY = -20.0
STEP_PENALTY = -1.0
GOAL_REWARD  = 50.0


# Instantiate and verify
env_grid = FireGridWorldEnv(
    grid_size  = GRID_SIZE,
    start      = START_STATE,
    goal       = GOAL_STATE,
    fire_cells = FIRE_CELLS,
    step_penalty = STEP_PENALTY,
    fire_penalty = FIRE_PENALTY,
    goal_reward = GOAL_REWARD,
    max_steps  = 300,
)

obs, _ = env_grid.reset(seed=SEED)
print(f'Observation space : {env_grid.observation_space}')
print(f'Action space      : {env_grid.action_space}   '
      f'(0=up, 1=down, 2=left, 3=right)')
print(f'Initial observation: state id {obs}  (= row 0 * 7 + col 2)')
print()
print('Initial grid  (A=agent  S=start  F=fire  G=goal  .=empty):')
print(env_grid.render())

**Question 2**: There are four methods (beyond the private helper methods) that must be implemented to satisfy the Gymnasium contract: `__init__()`, `reset()`, `step()`, and `render()`. For each of these four methods, write a one-sentence description of its purpose in the Gymnasium API and how it is used by an RL algorithm during training.


```
ANSWER HERE
```


**Question 3**: Notice that in `step()`, entering a fire cell gives `reward = -20.0` but `terminated = False` - the episode continues. Why is this a deliberate design choice? What would happen to the agent's _learned behavior_ if we instead set `terminated = True` when entering a fire cell?


```
ANSWER HERE
```


### Step A.2: Tabular Q-Learning on the Custom Environment

Because the fire grid has a small discrete state space (49 states, 4 actions), tabular Q-learning works perfectly. The cell below runs 350 episodes of Q-learning using the Gymnasium API (`reset`, `step`, `terminated`, `truncated`) exactly as any SB3 algorithm would, then visualizes the learned greedy policy.


In [ ]:
np.random.seed(SEED)

# For tabular Q-learning, we need to know the number of states and
# actions.
n_states  = env_grid.observation_space.n   # 49
n_actions = env_grid.action_space.n        # 4

# Q-table initialized with small random values to break symmetry
Q = np.random.uniform(-1, 1, size=(n_states, n_actions))

alpha   = 0.2    # learning rate
gamma   = 0.9    # discount factor
epsilon = 0.1    # epsilon-greedy exploration probability
n_episodes = 350

# We will track the return (total reward) of each episode to visualize 
# learning progress.
episode_returns = []

for ep in range(n_episodes):
    obs, _ = env_grid.reset()
    done = False
    ep_return = 0.0

    while not done:
        # Epsilon-greedy action selection
        if np.random.rand() < epsilon:
            action = env_grid.action_space.sample()   # explore
        else:
            action = int(np.argmax(Q[obs]))           # exploit

        next_obs, reward, terminated, truncated, _ = env_grid.step(action)
        done = terminated or truncated

        # Q-learning (off-policy): update toward max next-state value
        Q[obs, action] += alpha * (
            reward + gamma * np.max(Q[next_obs]) - Q[obs, action]
        )

        obs = next_obs
        ep_return += reward

    episode_returns.append(ep_return)

print(f'Training complete ({n_episodes} episodes)')
print(f'First 50-episode mean return: {np.mean(episode_returns[:50]):.1f}')
print(f'Last  50-episode mean return: {np.mean(episode_returns[-50:]):.1f}')

# Learning curve
window = 20
rolling = np.convolve(episode_returns, np.ones(window) / window, mode='valid')
plt.figure(figsize=(8, 3))
plt.plot(episode_returns, alpha=0.3, color='steelblue', label='Episode return')
plt.plot(np.arange(window - 1, len(episode_returns)), rolling,
         color='steelblue', linewidth=2, label=f'Rolling mean ({window} ep)')
plt.xlabel('Episode')
plt.ylabel('Return')
plt.title('Tabular Q-Learning on FireGridWorld (350 episodes)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def visualize_policy(env, Q, title=None):
    # Get environment parameters needed for visualization
    grid_size   = env.grid_size
    goal_state  = env.goal
    fire_cells  = env.fire_cells
    start_state = env.start

    # Visualize the greedy policy as a grid of Unicode arrows
    # (up down left right)
    arrow_map = {0: '↑', 1: '↓', 2: '←', 3: '→'}
    rows, cols = grid_size

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(-0.5, cols - 0.5)
    ax.set_ylim(rows - 0.5, -0.5)     # row 0 at the top
    ax.set_xticks(range(cols))
    ax.set_yticks(range(rows))
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    if title:
        ax.set_title(title)
    else:
        title = f'Greedy Policy After All Episodes'
    ax.set_title(title)
    ax.set_xticks(np.arange(-0.5, cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, rows, 1), minor=True)
    ax.grid(which='minor', color='gray', linewidth=0.8)
    ax.tick_params(which='minor', length=0)

    for r in range(rows):
        for c in range(cols):
            pos = (r, c)
            s   = r * cols + c

            # Always show the arrow (greedy policy action) in the center
            action = int(np.argmax(Q[s]))
            arrow = arrow_map[action]
            ax.text(c, r, arrow, ha='center', va='center',
                    fontsize=15, color='black', fontweight='bold')

            # Show special cell labels offset to avoid overlap
            if pos == goal_state:
                ax.text(c - 0.25, r - 0.25, 'G', ha='center', va='center',
                       fontsize=12, color='green', fontweight='bold')
            elif pos in fire_cells:
                ax.text(c - 0.25, r - 0.25, 'F', ha='center', va='center',
                       fontsize=12, color='red', fontweight='bold')
            elif pos == start_state:
                ax.text(c - 0.25, r - 0.25, 'S', ha='center', va='center',
                       fontsize=12, color='blue', fontweight='bold')

visualize_policy(env_grid, Q)

**Question 4**: Examine the greedy policy arrow grid. Trace the path the agent would follow from S to G by following the arrows. CAREFULLY answer the following:

1. Does the policy route around the fire columns, or does it go through them? (Keep in mind that there is a 10% probability of taking a random action at each step. When between the fires, 2 of those actions (right or left) might lead the agent into fire, so the agent must consider that it might accidentally step into fire and learn to route around it to reliably reach the goal.)
2. Approximately how many steps does the path take, and what is the expected return for that path (each normal step costs $-1$, the goal gives $+50$)? Is there a shorter path possible through the fire that is being avoided?


```
ANSWER HERE
```


### Step A.3: Effect of Fire Penalty - Student Coding Exercise

The `FireGridWorldEnv` constructor accepts a `fire_penalty` parameter that controls the penalty for stepping into the fire. The default is $-20.0$.

Right now, if the agent steps between the fires during its training, there is 10% chance of a random action instead of following the policy. But only 2 of the 4 possible actions (right or left) might lead the agent into fire when between the fires. This means only a 5% each step between the fire of accidentally stepping into fire. This means a total of a 15% chance of stepping into fire during any episode that happens to go through the fire after it figures out the optimal policy. Thus, the shortest path is still preferred unless the fire penalty is made more severe.

If we make the fire penalty smaller (e.g., -0.1), the agent may learn to go through fire more readily, since the cost of stepping into fire is not much worse than a normal step. **NOTE**: It turns out that increasing the fire penalty to something more severe (e.g., -50) there will be a stronger tendency to avoid the fire, but it does not change the greedy learned policy, since the greedy policy already identifies the shortest path through fire as optimal (since the risk of fire during any one training episode is only 15%, the expected return of the path through fire is still higher than the detour, so the same policy is learned).

Below we create a `run_q_learning` function which runs tabular Q-learning on any `FireGridWorldEnv` instance and returns the list of episode returns.


In [ ]:
def run_q_learning(env, n_episodes=350, alpha=0.2, gamma=0.9, epsilon=0.1,
                   seed=SEED):
    """Run tabular Q-learning on a FireGridWorldEnv;
    return per-episode returns and the learned Q-table."""
    np.random.seed(seed)
    Q = np.random.uniform(-1, 1,
                          size=(env.observation_space.n, env.action_space.n))
    returns = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        ep_return = 0.0
        while not done:
            action = (env.action_space.sample() if np.random.rand() < epsilon
                      else int(np.argmax(Q[obs])))
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            Q[obs, action] += alpha * (
                reward + gamma * np.max(Q[next_obs]) - Q[obs, action]
            )
            obs = next_obs
            ep_return += reward
        returns.append(ep_return)
    return returns, Q


print('run_q_learning helper function now ready.')

**Question 5 (coding)**: (a) Create two `FireGridWorldEnv` instances:

1. one with the default `fire_penalty=-20.0`(where `default_penalty = -20.0`) and
2. one with `fire_penalty=-0.5` (where `tiny_penalty = -0.5`).

and `run_q_learning` on each for 350 episodes, then run the two cells below to plot both rolling-mean learning curves on the same axes and the greedy policy for the tiny fire penalty.


In [ ]:
# Define two different fire penalties to compare
default_penalty = -20.0
tiny_penalty = -0.5

# TODO: Create two `FireGridWorldEnv` instances:
# 1. one with the default `fire_penalty=default_penalty` and
# 2. one with `fire_penalty=tiny_penalty`.
# Remember, you don't need to specify the other parameters since they
# will take the default values we set in the constructor.
env_small = ...
env_large = ...

# TODO: Run `run_q_learning` on each environment
returns_small, Q_small = ...
returns_large, Q_large = ...

# TODO Print the mean return of the last 50 episodes for each environment
# to compare final performance (Look a couple of cells up for one example
# of how to do this)


Plot the rolling mean learning curves for both environments on the same axes to compare how the fire penalty affects learning speed and stability.


In [ ]:
# Plot both rolling-mean learning curves on the same axes
window = 20
plt.figure(figsize=(9, 4))
for returns, label, color in [
    (returns_small, f'fire_penalty = {default_penalty}', 'steelblue'),
    (returns_large, f'fire_penalty = {tiny_penalty}', 'darkorange'),
]:
    roll = np.convolve(returns, np.ones(window) / window, mode='valid')
    plt.plot(returns, alpha=0.2, color=color)
    plt.plot(np.arange(window - 1, len(returns)), roll,
             color=color, linewidth=2, label=label)

plt.xlabel('Episode')
plt.ylabel('Return')
plt.title('Effect of Fire Penalty on Q-Learning in FireGridWorld')
plt.legend()
plt.tight_layout()
plt.show()


Plot the greedy policy for the tiny fire penalty to see how it differs from the greedy policy learned with the larger fire penalty. Does it differ? Why or why not?


In [ ]:
# Visualize the final greedy policy for the tiny fire penalty.
visualize_policy(env_small, Q_small,
                 title=f'Greedy Policy with fire_penalty = {tiny_penalty}')

**Question 5 (analysis)**: (b) Examine the output you generated and answer the following questions:

1. How does the tiny fire penalty change the agent's training process (as seen by its mean final 50 episode returns and the learning curve plot)?
2. How does the tiny fire penalty change the agent's behavior (as seen in its greedy policy)?


```
ANSWER HERE
```


## Part B: DQN on CartPole - Effect of Exploration Fraction

### Background

In lecture we also discussed the CartPole environment and the DQN algorithm. CartPole's 4-dimensional _continuous_ state space makes tabular Q-learning impractical (infinitely many possible states). SB3's `DQN` replaces the Q-table with a neural network (`MlpPolicy`) that takes the state vector as input and outputs a Q-value for each discrete action. SB3 handles the neural network architecture, experience replay buffer, and target network updates automatically.

The hyperparameter we study is **`exploration_fraction`** - the fraction of total training timesteps over which $\epsilon$ (the exploration probability) is annealed from 1.0 down to `exploration_final_eps`. This directly controls the exploration-exploitation tradeoff from lecture:

- **Too small**: epsilon drops to its minimum almost immediately. The agent barely explores and gets stuck exploiting the first (probably bad) behaviors it stumbles upon.
- **Balanced**: the agent explores long enough to discover useful state-action relationships before shifting to exploitation.
- **Too large**: the agent spends most of training acting randomly and has little time to exploit what it has learned.

### Why CartPole is Not the Best Toy Environment for Studying Exploration

It turns out the optimal policy for CartPole is actually quite easy to learn (e.g. go in the direction the pole is falling), and the environment is not very sensitive to the exploration fraction. Even with a very small exploration fraction, the agent can quickly discover a good policy that balances the pole for hundreds of steps. I used too large an exploration fraction in lecture (around 0.3), which is typical in other problems but unnecessarily large for CartPole.

We can still see the effect of exploration fraction on learning speed and stability, but the differences are not as dramatic as they would be in a more complex environment.

### Step B.1: Guided Example - One DQN Run

Start by looking at the `train_dqn()` function defined below, specifically the call to `DQN` and the `model.learn()` method. This is the standard way to train any SB3 algorithm on any Gymnasium environment.


In [ ]:
N_TIMESTEPS_DQN = 25_000

def train_dqn(exploration_fraction, label,
              n_timesteps=N_TIMESTEPS_DQN, seed=SEED):
    """
    Train a DQN agent on CartPole-v1 and return (episode_rewards, model).

    Parameters
    ----------
    exploration_fraction : float
        Fraction of n_timesteps over which epsilon is annealed.
    label : str
        Human-readable label printed with the result summary.
    """
    env = Monitor(gym.make('CartPole-v1'))
    model = DQN(
        'MlpPolicy', env,
        verbose=0,
        seed=seed,
        # --- hyperparameter under study ---
        exploration_fraction=exploration_fraction,
        # --- fixed hyperparameters ---
        learning_rate=1e-3,          # Adam optimizer step size
        exploration_final_eps=0.02,  # epsilon floor after annealing
        learning_starts=500,         # random steps before network training begins
        gamma=0.99,                  # discount factor
        batch_size=64,               # mini-batch size sampled from replay buffer
        buffer_size=10_000,          # replay buffer capacity
        target_update_interval=500,  # steps between target network syncs
    )
    model.learn(total_timesteps=n_timesteps)
    rewards = env.get_episode_rewards()
    env.close()
    print(f'  {label}: {len(rewards):4d} episodes, '
          f'last-30 mean = {np.mean(rewards[-30:]):.0f}')
    return rewards, model


def plot_learning_curves(rewards_dict, title, max_reward=500, window=40):
    """Plot rolling-mean learning curves for one or more runs."""
    fig, ax = plt.subplots(figsize=(10, 5))
    for label, (rewards, color) in rewards_dict.items():
        w = min(window, max(5, len(rewards) // 8))
        roll = np.convolve(rewards, np.ones(w) / w, mode='valid')
        ax.plot(rewards, alpha=0.15, color=color)
        ax.plot(np.arange(w - 1, len(rewards)), roll,
                color=color, linewidth=2, label=label)
    if max_reward:
        ax.axhline(max_reward, color='green', linestyle='--',
                   linewidth=0.8, label=f'Max ({max_reward})')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Episode Reward')
    ax.set_title(title)
    ax.legend(loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
# Run a DQN agent with a balanced exploration schedule (epsilon annealed over 5% 
# of training) NOTE: This value is smaller than most more complicated
# environments, where 10-20% is more common.
rewards_med, model_med = train_dqn(0.05, 'exploration_fraction = 0.05  (balanced)')

plot_learning_curves(
    {'balanced (0.05)': (rewards_med, 'steelblue')},
    title='DQN Learning Curve with Balanced Exploration Schedule',
)

**Question 6**: Looking at the balanced DQN learning curve, describe its general shape. Is there a region of low performance before improvement begins? What is the agent doing during the `learning_starts=500` timesteps at the very start, and why does performance tend to pick up only after that point? **HINT**: Before `learning_starts`, the agent is taking random actions and filling the replay buffer with experience, but it is not yet updating the neural network policy.`


```
ANSWER HERE
```


### Step B.2: Three Exploration Fractions - Comparison Experiment

Let's now compare two other exploration scenarios with different `exploration_fraction` values to see how this hyperparameter shapes the learning curve and final performance.


In [ ]:
print(f'Training remaining DQN agents ({N_TIMESTEPS_DQN:,} timesteps each)...')
print()
rewards_low,  model_low  = train_dqn(0.01, 'exploration_fraction = 0.01  (barely explore)')
rewards_high, model_high = train_dqn(0.30, 'exploration_fraction = 0.30  (over-explore)')

print()
print('Summary  (last-30 episode mean):')
for lbl, r in [('ef=0.01', rewards_low),
               ('ef=0.05', rewards_med),
               ('ef=0.30', rewards_high)]:
    print(f'  {lbl}:  {np.mean(r[-30:]):.0f}')

plot_learning_curves(
    {
        'ef = 0.05   barely explore': (rewards_low,  'tomato'),
        'ef = 0.30   balanced':       (rewards_med,  'steelblue'),
        'ef = 0.95   over-explore':   (rewards_high, 'darkorange'),
    },
    title=f'DQN CartPole: Effect of exploration_fraction ({N_TIMESTEPS_DQN:,} timesteps)'
)

**Question 7**: Examine the three learning curves and the last-30 episode means. Describe the behavior of each agent:

- **ef=0.01** (barely explore): Does performance improve quickly? Does it reach high performance by the end? Why or why not?
- **ef=0.05** (balanced): How does it compare to the other two?
- **ef=0.30** (over-explore): Why does it tend to produce more episodes than ef=0.01 for the same timestep budget?


```
ANSWER HERE
```


`evaluate_policy` is a built-in SB3 function that runs a trained model for a specified number of episodes and returns the mean and standard deviation of the episode rewards. If you pass it `deterministic=True`, it will evaluate the greedy policy without exploration noise, giving a more accurate measure of the learned policy's performance.

**Question 8 (Coding)**: The following cell uses `evaluate_policy` with `deterministic=True` and `n_eval_episodes=100` to measure the final performance of all three trained models. Print the mean and standard deviation for each. Then answer: does the ranking by `evaluate_policy` match the ranking by last-30 training mean? Why is `evaluate_policy` a more reliable performance measure than the end of the training curve?


In [ ]:
print('evaluate_policy results (deterministic=True, 100 episodes):')
for label, model in [
    ('ef=0.01 (barely explore)', model_low),
    ('ef=0.05 (balanced)',       model_med),
    ('ef=0.30 (over-explore)',   model_high),
]:
    eval_env = Monitor(gym.make('CartPole-v1'))

    # 
    mean_r, std_r = evaluate_policy(
        model, eval_env, n_eval_episodes=100, deterministic=True
    )
    eval_env.close()
    # TODO: Print the mean and standard deviation of the returns for each model

```
ANSWER HERE
```


## Part C: PPO on CartPole - Effect of the Discount Factor $\gamma$

### Background

**Proximal Policy Optimization (PPO)** directly learns the policy $\pi_{\theta}(a \mid s)$ as a neural network, adjusted by gradient ascent on the expected return. Unlike DQN it does not learn a Q-function first. PPO is **on-policy**: it collects a batch of experience using its current policy, performs several gradient updates on that batch, then discards it and collects a fresh batch. This means each environment interaction is used only once, but updates are more stable than naive policy gradient methods.

The hyperparameter we study is **`gamma`** ($\gamma$), the discount factor in:

$$G_{\gamma} = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \ldots + \gamma^{e-t} r_e$$

- **Low $\gamma$**: future rewards are heavily discounted. The agent is short-sighted.
- **High $\gamma$**: future rewards are nearly as valuable as immediate ones. The agent plans further ahead.

CartPole gives $+1$ every timestep, so good performance requires sustaining many future rewards - a high $\gamma$ is essential. This makes it ideal for demonstrating what happens when $\gamma$ is too low.

### Step C.1: Guided Example - Balanced Gamma


In [ ]:
N_TIMESTEPS_PPO = 30_000


def train_ppo(gamma, label, n_timesteps=N_TIMESTEPS_PPO, seed=SEED):
    """
    Train a PPO agent on CartPole-v1 and return (episode_rewards, model).

    Parameters
    ----------
    gamma : float
        Discount factor in [0, 1].
    label : str
        Human-readable label printed with the result summary.
    """
    env = Monitor(gym.make('CartPole-v1'))
    model = PPO(
        'MlpPolicy', env,
        verbose=0,
        seed=seed,
        # --- hyperparameter under study ---
        gamma=gamma,
        # --- fixed hyperparameters ---
        n_steps=512,          # timesteps collected per rollout before each update
        batch_size=64,        # mini-batch size for gradient updates
        learning_rate=3e-4,   # Adam optimizer step size
        gae_lambda=0.95,      # lambda for Generalized Advantage Estimation
    )
    model.learn(total_timesteps=n_timesteps)
    rewards = env.get_episode_rewards()
    env.close()
    print(f'  {label}: {len(rewards):4d} episodes, '
          f'last-20 mean = {np.mean(rewards[-20:]):.0f}')
    return rewards, model


print(f'Training PPO (gamma=0.95) on CartPole-v1 ({N_TIMESTEPS_PPO:,} timesteps)...')
rewards_g095, model_g095 = train_ppo(gamma=0.95, label='gamma = 0.95  (balanced)')

# Plot the learning curve for the PPO agent with gamma=0.95
plot_learning_curves(
    {'gamma = 0.95  balanced': (rewards_g095, 'steelblue')},
    title=f'PPO CartPole: Balanced Discount Factor ({N_TIMESTEPS_PPO:,} timesteps)'
)

**Question 9**: The PPO learning curve for `gamma=0.95` almost certainly shows far fewer episodes than the DQN curves in Part B, despite a similar timestep budget. Explain why this is a _good_ sign rather than a bad one. (Hint: think about what happens to episode length as PPO's policy improves, and contrast that with DQN's episode lengths early in training when epsilon is high.)


```
ANSWER HERE
```


### Step C.2: Three Gamma Values - Comparison Experiment

So increasing gamma is biasing the agent toward valuing future rewards more. In cartpole, this is essential because the reward is +1 per step. Let's compare three gamma values to see how this hyperparameter shapes learning and final performance.


In [ ]:
print(f'Training three PPO agents ({N_TIMESTEPS_PPO:,} timesteps each)...')
print()
rewards_g05,   model_g05   = train_ppo(0.5,   'gamma = 0.50   (very shortsighted)')
# gamma=0.95 already trained above
rewards_g0999, model_g0999 = train_ppo(0.999, 'gamma = 0.999  (very farsighted)')

print()
print('Summary  (last-20 episode mean):')
for lbl, r in [('gamma=0.500', rewards_g05),
               ('gamma=0.950', rewards_g095),
               ('gamma=0.999', rewards_g0999)]:
    print(f'  {lbl}:  {np.mean(r[-20:]):.0f}')

# Plot all three learning curves on the same axes for comparison
plot_learning_curves(
    {
        'gamma = 0.500   shortsighted': (rewards_g05,   'tomato'),
        'gamma = 0.950   balanced':     (rewards_g095,  'steelblue'),
        'gamma = 0.999   farsighted':   (rewards_g0999, 'darkorange'),
    },
    title=f'PPO CartPole: Effect of Discount Factor gamma ({N_TIMESTEPS_PPO:,} timesteps)'
)

**Question 10**: Examine the three PPO learning curves and the last-20 episode means. For each setting, connect the observed performance to the mathematical definition of discounted return:

- **gamma=0.50** (shortsighted): The discounted return formula shows that with gamma=0.5, a reward 10 steps in the future is worth only 0.5^10 = 0.00098 of its original value, which is negligible. given this fact, what does the agent effectively optimize? Why is that a problem for keeping a pole balanced for 500 steps?
- **gamma=0.95** (balanced): Roughly how many steps ahead does this agent effectively plan?
- **gamma=0.999** (farsighted): In essence, all rewards are considered to be somewhat significant, even ones that are too far in the future to actually be affected by the agent's actions. Why would this HURT the performance compared gamma=0.95? **HINT**: If the future reward are not actually relevant, what can the agent learn from them?


```
ANSWER HERE
```


### Step C.3: Saving, Evaluating, and Visualizing the Best PPO Model

Just so you have an example of it, the cells below show how to:

1. save a trained SB3 model,
2. load the model,
3. evaluate it with `evaluate_policy`, and
4. visualize its behavior by rendering a video of the agent acting in the environment.

The video is saved as an MP4 file that you can download and watch on your computer. You can also render the video directly in Colab using `HTML` and `base64` encoding, but that is a bit more complex so I have not included it here.


In [ ]:
# Save the best model and verify the save/load cycle
model_g095.save('lab11_cartpole_ppo_best')
print('Model saved to lab11_cartpole_ppo_best.zip')

loaded = PPO.load('lab11_cartpole_ppo_best')
print('Model loaded successfully.')

# Deterministic evaluation
eval_env = Monitor(gym.make('CartPole-v1'))
mean_r, std_r = evaluate_policy(
    loaded, eval_env, n_eval_episodes=100, deterministic=True
)
eval_env.close()

print(f'\nEvaluation over 100 episodes (deterministic policy):')
print(f'  Mean reward : {mean_r:.1f}  +/-  {std_r:.1f}')
print(f'  Max possible: 500')

**Question 11**: Compare the `evaluate_policy` mean reward for the gamma=0.95 PPO model to its last-20 training mean.

1. Is the mean reward for the last 20 training episodes lower than the `evaluate_policy` mean? Why might that be the case?
2. Based on what you learned in Parts B and C, explain in your own words why `evaluate_policy` with `deterministic=True` gives a more trustworthy measure of the final policy's quality than simply reading the end of the training curve.


```
ANSWER HERE
```


And now the video of the motion of the gamma=0.95 PPO agent in CartPole. You should see the pole balancing for hundreds of steps, demonstrating the high performance achieved by this setting of the discount factor.


In [ ]:
# Record one episode and display it as an animation
import matplotlib.animation as animation
from IPython.display import Image

vis_env = gym.make('CartPole-v1', render_mode='rgb_array')
obs, _ = vis_env.reset(seed=0)
frames = []
total_r = 0.0
done = False

while not done:
    frames.append(vis_env.render())
    action, _ = loaded.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = vis_env.step(action)
    total_r += reward
    done = terminated or truncated

vis_env.close()
print(f'Recorded episode: {len(frames)} steps, total reward = {total_r:.0f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.axis('off')
im = ax.imshow(frames[0])

# Add frame counter text
frame_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, 
                     fontsize=12, color='white', fontweight='bold',
                     bbox=dict(boxstyle='round', facecolor='black', alpha=0.7),
                     verticalalignment='top')

def update_frame(i):
    im.set_array(frames[i])
    frame_text.set_text(f'Frame: {i+1}/{len(frames)}')
    return [im, frame_text]

anim = animation.FuncAnimation(
    fig, update_frame, frames=len(frames), interval=33, blit=True
)

gif_path = 'lab11_cartpole_ppo.gif'
try:
    anim.save(gif_path, writer='pillow', fps=30)
    plt.close(fig)
    display(Image(gif_path))
    print(f'Animation saved as {gif_path}')
except Exception as exc:
    plt.close(fig)
    print(f'GIF export failed ({exc}); trying HTML animation...')
    from IPython.display import HTML
    display(HTML(anim.to_jshtml()))

**Question 12**: (You will likely have to review the class lecture notes/textbook to answer this one) Across the lab you have used three different approaches to the same type of task: tabular Q-learning on FireGridWorld (Part A), DQN on CartPole (Part B), and PPO on CartPole (Part C). Summarize the key differences along these two dimensions:

(a) What type of state space does each method require (discrete vs. continuous), and why?

(b) Is each method on-policy or off-policy, and what practical consequence does that have for how efficiently each method uses environment interactions?


```
ANSWER HERE
```


## What You Need to Submit

Once you have completed this lab, go to the File menu and select **Download > Download .ipynb**. Submit the downloaded file to the Lab 11 dropbox on D2L.
